# Notebook 08: Final Integration - Unified Travel Agent

## Learning Objectives
- Integrate all AgentCore components into a single unified agent
- Reuse existing resources from previous chapters
- Deploy production-ready travel companion
- Test complete end-to-end workflow

## Prerequisites
- Completed Notebooks 02-07
- Existing Gateway, Memory, and Cognito resources
- All component configurations saved

This notebook runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.

## Step 1: Setup and Resource Discovery

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");
// Remove any existing credential env vars to force profile usage
// for (const key of ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]) {
//   Deno.env.delete(key);
// }

console.log("\u2705 AWS Profile set. Please restart kernel and run all cells.");

In [ ]:
import type { CognitoClientInfo } from "../toolkit/mod.ts";
import { loadEnv, maskKey, state, writeFile } from "../shared/notebook.ts";

// Load environment variables
await loadEnv();

const REGION = Deno.env.get("AWS_REGION") ?? "us-east-1";

// The shapes the earlier notebooks wrote into environments/
interface GatewayInfo {
  gateway_name: string;
  gateway_id: string;
  mcp_endpoint: string;
  region: string;
  oauth_client_id: string;
  oauth_client_secret: string;
  oauth_scope: string;
  available_tools: string[];
  token_endpoint?: string;
}

interface MemoryInfo {
  memory_id: string;
  memory_name: string;
  region: string;
  user_id: string;
  session_id: string;
}

interface RuntimeInfo {
  agent_name: string;
  agent_arn: string;
  agent_id: string;
  ecr_uri: string;
  region: string;
  status: string;
}

interface CognitoConfig {
  gateway_name: string;
  authorizer_config: Record<string, unknown>;
  client_info: CognitoClientInfo;
}

/** Load an existing resource configuration, naming the notebook that writes it. */
async function loadResourceConfig<T>(filename: string, createdBy: string): Promise<T> {
  try {
    return JSON.parse(await Deno.readTextFile(`environments/${filename}`)) as T;
  } catch {
    throw new Error(`Missing environments/${filename} \u2014 run ${createdBy} first.`);
  }
}

// Load existing resources
const gatewayInfo = await loadResourceConfig<GatewayInfo>(
  "gateway_info.json",
  "notebook 03 (Gateway Integration)",
);
const memoryInfo = await loadResourceConfig<MemoryInfo>(
  "memory_info.json",
  "notebook 04 (Memory Implementation)",
);
const runtimeInfo = await loadResourceConfig<RuntimeInfo>(
  "runtime_info.json",
  "notebook 02 (Runtime Setup)",
);
const cognitoConfig = await loadResourceConfig<CognitoConfig>(
  "cognito_config.json",
  "notebook 03 (Gateway Integration)",
);

console.log("\ud83d\udccb Resource Discovery:");
console.log(`\u2705 Gateway: ${gatewayInfo.gateway_id}`);
console.log(`\u2705 Memory: ${memoryInfo.memory_id}`);
console.log(`\u2705 Runtime: ${runtimeInfo.agent_name} (${runtimeInfo.status})`);
console.log(`\u2705 Cognito: ${cognitoConfig.client_info.client_id.slice(0, 10)}...`);

## Step 1.4: Ensure OAuth Configuration

In [ ]:
// Ensure the Cognito client has the client_credentials OAuth flow the Gateway needs
import { activateOauthClientCredentials } from "../backend/cognito_config.ts";

console.log("\ud83d\udd10 Verifying OAuth Configuration...");
console.log("=".repeat(60));

await activateOauthClientCredentials(cognitoConfig.client_info, REGION);

## Step 1.5: Test Gateway Connectivity

In [ ]:
// The ExchangeRate spec takes its key as a request parameter, so read it back from the credential
// provider the Gateway created. `IdentityHelper` is the same module the deployed agent uses, so
// the notebook and the runtime read the key exactly the same way (notebook 03 does this inline).
import { IdentityHelper } from "../backend/runtime/final_agent/identity_helper.ts";

const identityHelper = new IdentityHelper(REGION);

console.log("\ud83d\udd0d Searching for ExchangeRate API key provider...");
const exchangeRateApikey = await identityHelper.getExchangerateApiKey();

if (exchangeRateApikey) {
  console.log("\u2705 Successfully retrieved API key");
  // Masked on purpose: notebook outputs get committed
  console.log(`\n\ud83c\udf89 ExchangeRate API Key: ${maskKey(exchangeRateApikey)}`);
} else {
  console.log("\u274c Error retrieving API key");
}

In [ ]:
// JSON-RPC shapes the MCP endpoint answers with
interface McpTool {
  name?: string;
}

interface McpResult {
  result?: { tools?: McpTool[]; content?: { text?: string }[] };
  error?: unknown;
}

/** Test OAuth token retrieval from Cognito. */
async function testOauthToken(): Promise<string | null> {
  console.log("\ud83d\udd10 Testing OAuth Token Retrieval...");
  console.log("-".repeat(60));

  // The token endpoint comes from the Cognito config saved in notebook 03
  const tokenEndpoint = cognitoConfig.client_info.token_endpoint;

  console.log(`Token Endpoint: ${tokenEndpoint}`);
  console.log(`Client ID: ${gatewayInfo.oauth_client_id}`);
  console.log(`Scope: ${gatewayInfo.oauth_scope}`);

  try {
    const response = await fetch(tokenEndpoint, {
      method: "POST",
      headers: { "Content-Type": "application/x-www-form-urlencoded" },
      body: new URLSearchParams({
        grant_type: "client_credentials",
        client_id: gatewayInfo.oauth_client_id,
        client_secret: gatewayInfo.oauth_client_secret,
        scope: gatewayInfo.oauth_scope,
      }),
    });

    console.log(`\nStatus Code: ${response.status}`);

    if (!response.ok) {
      console.log("\u274c Token retrieval failed!");
      console.log(`Response: ${await response.text()}`);
      return null;
    }

    const tokenData = await response.json() as {
      access_token?: string;
      token_type?: string;
      expires_in?: number;
    };
    console.log("\u2705 Token retrieved successfully!");
    console.log(`Token Type: ${tokenData.token_type}`);
    console.log(`Expires In: ${tokenData.expires_in} seconds`);
    console.log(`Access Token (first 20 chars): ${(tokenData.access_token ?? "").slice(0, 20)}...`);
    return tokenData.access_token ?? null;
  } catch (error) {
    console.log(`\u274c Error: ${error}`);
    return null;
  }
}

/** Test MCP endpoint connectivity with a `tools/list` call. */
async function testMcpEndpoint(accessToken: string | null): Promise<boolean> {
  console.log("\n\ud83d\udd0c Testing MCP Endpoint Connectivity...");
  console.log("-".repeat(60));

  const mcpEndpoint = gatewayInfo.mcp_endpoint;
  console.log(`MCP Endpoint: ${mcpEndpoint}`);

  if (!accessToken) {
    console.log("\u274c No access token available. Skipping MCP test.");
    return false;
  }

  try {
    const response = await fetch(mcpEndpoint, {
      method: "POST",
      headers: { Authorization: `Bearer ${accessToken}`, "Content-Type": "application/json" },
      body: JSON.stringify({
        jsonrpc: "2.0",
        id: "test-list-tools",
        method: "tools/list",
        params: {},
      }),
    });

    console.log(`\nStatus Code: ${response.status}`);

    if (!response.ok) {
      console.log("\u274c MCP endpoint test failed!");
      console.log(`Response: ${await response.text()}`);
      return false;
    }

    const result = await response.json() as McpResult;
    console.log("\u2705 MCP endpoint is accessible!");

    const tools = result.result?.tools;
    if (tools) {
      console.log(`\n\ud83d\udccb Available Tools (${tools.length}):`);
      for (const tool of tools) console.log(`  \u2022 ${tool.name ?? "Unknown"}`);
    } else {
      console.log(`Response: ${JSON.stringify(result, null, 2)}`);
    }
    return true;
  } catch (error) {
    console.log(`\u274c Error: ${error}`);
    return false;
  }
}

/**
 * Test a specific gateway tool call over MCP (JSON-RPC over HTTP), replacing Python's `requests`.
 * Upstream errors are printed, never thrown: with placeholder API keys the APIs answer 401 and the
 * notebook should carry on.
 */
async function testGatewayToolCall(
  accessToken: string | null,
  toolName: string,
  args: Record<string, unknown>,
): Promise<McpResult | null> {
  console.log(`\n\ud83d\udee0\ufe0f Testing Tool: ${toolName}`);
  console.log("-".repeat(60));

  if (!accessToken) {
    console.log("\u274c No access token available. Skipping tool test.");
    return null;
  }

  try {
    const response = await fetch(gatewayInfo.mcp_endpoint, {
      method: "POST",
      headers: { Authorization: `Bearer ${accessToken}`, "Content-Type": "application/json" },
      body: JSON.stringify({
        jsonrpc: "2.0",
        id: `test-${toolName}`,
        method: "tools/call",
        params: { name: toolName, arguments: args },
      }),
    });

    console.log(`Arguments: ${JSON.stringify(args, null, 2)}`);
    console.log(`Status Code: ${response.status}`);

    if (!response.ok) {
      console.log("\u274c Tool call failed!");
      console.log(`Response: ${await response.text()}`);
      return null;
    }

    const result = await response.json() as McpResult;
    console.log("\u2705 Tool call successful!");
    console.log("\nResult:");
    console.log(JSON.stringify(result, null, 2));
    return result;
  } catch (error) {
    console.log(`\u274c Error: ${error}`);
    return null;
  }
}

// Run all gateway tests
console.log("\ud83e\uddea GATEWAY CONNECTIVITY TESTS");
console.log("=".repeat(60));
console.log(`Gateway ID: ${gatewayInfo.gateway_id}`);
console.log(`Region: ${gatewayInfo.region}`);
console.log();

// Test 1: OAuth Token
const accessToken = await testOauthToken();

// Test 2: MCP Endpoint
if (accessToken) {
  const mcpSuccess = await testMcpEndpoint(accessToken);

  // Test 3: Sample Tool Calls
  if (mcpSuccess) {
    console.log(`\n${"=".repeat(60)}`);
    console.log("Testing Sample Tool Calls");
    console.log("=".repeat(60));

    // Test flight search
    await testGatewayToolCall(accessToken, "FlightSearch___getFlights", {
      dep_iata: "NYC",
      arr_iata: "LAX",
    });

    // Test weather
    await testGatewayToolCall(accessToken, "WeatherSearch___getCurrentWeather", {
      q: "Rome,IT",
      units: "metric",
    });

    // Test currency conversion
    await testGatewayToolCall(accessToken, "ExchangeRate___convertCurrency", {
      api_key: exchangeRateApikey,
      from_currency: "USD",
      to_currency: "EUR",
    });
  }
}

console.log(`\n${"=".repeat(60)}`);
console.log("\u2705 Gateway connectivity tests completed!");
console.log("=".repeat(60));

// Save token endpoint for later use
if (accessToken) {
  gatewayInfo.token_endpoint = cognitoConfig.client_info.token_endpoint;

  // Update gateway_info.json with token endpoint
  await writeFile("environments/gateway_info.json", `${JSON.stringify(gatewayInfo, null, 2)}\n`);
  console.log("\n\ud83d\udcbe Token endpoint saved to gateway_info.json");
}

In [ ]:
const flights = await testGatewayToolCall(accessToken, "FlightSearch___getFlights", {
  dep_iata: "FRA",
  arr_iata: "BER",
});

console.log(flights);

In [ ]:
// Test weather
await testGatewayToolCall(accessToken, "WeatherSearch___getCurrentWeather", {
  q: "Frankfurt,DE",
  units: "metric",
});

In [ ]:
// Test currency conversion, with the key read back from the credential provider above
await testGatewayToolCall(accessToken, "ExchangeRate___convertCurrency", {
  api_key: exchangeRateApikey,
  from_currency: "USD",
  to_currency: "EUR",
});

## Step 2: Create Unified Travel Agent

The three cells below write the deployable agent folder, replacing the `%%writefile` cells of the
Python course: the unified agent, its identity helper, and the `deno.json` that plays the role
`requirements.txt` played before.

In [ ]:
await writeFile("../backend/runtime/final_agent/unified_travel_agent.ts", `#!/usr/bin/env -S deno run -A
/**
 * Unified Travel Agent - Combines all AgentCore components
 * Integrates Gateway MCP, Memory, Code Interpreter, Browser Tools, and OAuth.
 * Uses environment variables for configuration (set during runtime deployment).
 *
 * Replaces \`unified_travel_agent.py\`. Tools are declared with \`tool({ inputSchema: zod })\`, the
 * Gateway is still reached by hand-rolled JSON-RPC over \`fetch\` (as the Python did, rather than
 * through an MCP client), and memory goes through the course's own \`MemoryClient\`.
 */
import { Agent, BedrockModel, tool } from "@strands-agents/sdk";
import { BedrockAgentCoreApp } from "bedrock-agentcore/runtime";
import { z } from "zod";
import { MemoryClient } from "../../../toolkit/mod.ts";
import { IdentityHelper } from "./identity_helper.ts";

// Configuration from environment variables
const REGION = Deno.env.get("AWS_REGION") ?? "us-east-1";
const MODEL_ID = Deno.env.get("MODEL_ID") ?? "us.anthropic.claude-haiku-4-5-20251001-v1:0";

// Gateway configuration from environment
const GATEWAY_MCP_ENDPOINT = Deno.env.get("GATEWAY_MCP_ENDPOINT");
const GATEWAY_TOKEN_ENDPOINT = Deno.env.get("GATEWAY_TOKEN_ENDPOINT");
const GATEWAY_OAUTH_CLIENT_ID = Deno.env.get("GATEWAY_OAUTH_CLIENT_ID");
const GATEWAY_OAUTH_CLIENT_SECRET = Deno.env.get("GATEWAY_OAUTH_CLIENT_SECRET");
const GATEWAY_OAUTH_SCOPE = Deno.env.get("GATEWAY_OAUTH_SCOPE");

// Memory configuration from environment
const MEMORY_ID = Deno.env.get("MEMORY_ID");
const MEMORY_USER_ID = Deno.env.get("MEMORY_USER_ID") ?? "default-user";
const MEMORY_SESSION_ID = Deno.env.get("MEMORY_SESSION_ID") ?? "default-session";

// Initialize tools
const memoryClient = MEMORY_ID ? new MemoryClient({ region: REGION }) : null;
const identityHelper = new IdentityHelper(REGION);

/** Internal helper to call MCP gateway tools. */
async function callMcpTool(toolName: string, args: Record<string, unknown>): Promise<string> {
  if (!GATEWAY_MCP_ENDPOINT) {
    return JSON.stringify({ error: "Gateway not configured" });
  }

  try {
    // Get access token
    const tokenResponse = await fetch(GATEWAY_TOKEN_ENDPOINT!, {
      method: "POST",
      headers: { "content-type": "application/x-www-form-urlencoded" },
      body: new URLSearchParams({
        grant_type: "client_credentials",
        client_id: GATEWAY_OAUTH_CLIENT_ID ?? "",
        client_secret: GATEWAY_OAUTH_CLIENT_SECRET ?? "",
        scope: GATEWAY_OAUTH_SCOPE ?? "",
      }),
    });

    if (!tokenResponse.ok) {
      return JSON.stringify({ error: "Authentication failed" });
    }

    const accessToken = (await tokenResponse.json()).access_token;

    // Call MCP endpoint
    const response = await fetch(GATEWAY_MCP_ENDPOINT, {
      method: "POST",
      headers: {
        Authorization: \`Bearer \${accessToken}\`,
        "Content-Type": "application/json",
      },
      body: JSON.stringify({
        jsonrpc: "2.0",
        id: \`unified-\${toolName}\`,
        method: "tools/call",
        params: { name: toolName, arguments: args },
      }),
    });

    if (!response.ok) {
      return JSON.stringify({ error: \`API call failed: \${response.status}\` });
    }
    const result = await response.json();
    return JSON.stringify(result.result ?? {});
  } catch (error) {
    return JSON.stringify({ error: \`Error calling gateway API: \${error}\` });
  }
}

const searchFlights = tool({
  name: "search_flights",
  description: "Search for flights between two airports",
  inputSchema: z.object({
    origin: z.string().describe("Departure IATA code (e.g., SFO)"),
    destination: z.string().describe("Arrival IATA code (e.g., DFW)"),
  }),
  callback: ({ origin, destination }) =>
    callMcpTool("FlightSearch___getFlights", { dep_iata: origin, arr_iata: destination }),
});

const getWeather = tool({
  name: "get_weather",
  description: "Get current weather for a location",
  inputSchema: z.object({
    location: z.string().describe("City name and country code (e.g., 'Rome,IT')"),
    units: z.string().default("metric").describe("'metric' (Celsius) or 'imperial' (Fahrenheit)"),
  }),
  callback: ({ location, units }) =>
    callMcpTool("WeatherSearch___getCurrentWeather", { q: location, units }),
});

const convertCurrency = tool({
  name: "convert_currency",
  description: "Get current exchange rate between two currencies",
  inputSchema: z.object({
    from_currency: z.string().describe("Source currency code (e.g., 'USD')"),
    to_currency: z.string().describe("Target currency code (e.g., 'EUR')"),
  }),
  callback: async ({ from_currency, to_currency }) => {
    // Retrieve API key from credential provider
    const apiKey = await identityHelper.getExchangerateApiKey();
    if (!apiKey) {
      return JSON.stringify({ error: "ExchangeRate API key not available" });
    }
    return await callMcpTool("ExchangeRate___convertCurrency", {
      api_key: apiKey,
      from_currency,
      to_currency,
    });
  },
});

const getUserPreferences = tool({
  name: "get_user_preferences",
  description: "Retrieve user travel preferences from memory",
  inputSchema: z.object({}),
  callback: async () => {
    if (!memoryClient || !MEMORY_ID) {
      return "Memory not configured - missing MEMORY_ID environment variable";
    }
    try {
      const memories = await memoryClient.retrieveMemories({
        memoryId: MEMORY_ID,
        namespace: \`travel/user/\${MEMORY_USER_ID}/preferences\`,
        query: "travel preferences",
        topK: 5,
      });
      const preferences = memories
        .map((m) => (m.content as { text?: string } | undefined)?.text)
        .filter((text): text is string => Boolean(text));
      return JSON.stringify({ preferences, user_id: MEMORY_USER_ID });
    } catch (error) {
      return \`Error retrieving preferences: \${error}\`;
    }
  },
});

const saveTravelMemory = tool({
  name: "save_travel_memory",
  description: "Save travel information to memory",
  inputSchema: z.object({
    content: z.string().describe("What to remember"),
    memory_type: z.string().default("semantic"),
  }),
  callback: async ({ content }) => {
    if (!memoryClient || !MEMORY_ID) {
      return "Memory not configured - missing MEMORY_ID environment variable";
    }
    try {
      await memoryClient.createEvent({
        memoryId: MEMORY_ID,
        actorId: MEMORY_USER_ID,
        sessionId: MEMORY_SESSION_ID,
        messages: [[content, "ASSISTANT"]],
      });
      return "Memory saved successfully";
    } catch (error) {
      return \`Error saving memory: \${error}\`;
    }
  },
});

// Create unified agent
const model = new BedrockModel({ modelId: MODEL_ID });

const unifiedAgent = new Agent({
  model,
  tools: [
    searchFlights,
    getWeather,
    convertCurrency,
    getUserPreferences,
    saveTravelMemory,
    // The code interpreter and browser tools stay commented out here, as in the Python original:
    // notebooks 06 and 07 exercise them, and each opens a billed session per agent instance.
  ],
  systemPrompt: \`
You are a comprehensive AI Travel Companion with access to:

1. **Flight Search**: search_flights(origin, destination) - Find flights between airports
2. **Weather**: get_weather(location, units) - Get current weather for a city
3. **Currency**: convert_currency(from_currency, to_currency) - Get exchange rates
4. **Memory**: get_user_preferences() and save_travel_memory(content) - Store/retrieve preferences

Always:
- Check user preferences first using get_user_preferences()
- Use real APIs for current flight, hotel, weather, and currency information
- Provide comprehensive travel planning with budget considerations
- Save important travel decisions to memory

Provide comprehensive, personalized travel planning assistance.
\`,
});

// Initialize AgentCore app
const app = new BedrockAgentCoreApp({
  invocationHandler: {
    requestSchema: z.object({
      prompt: z.string().default("Hello! How can I help you plan your travel?"),
    }),
    process: async ({ prompt }) => {
      try {
        const response = await unifiedAgent.invoke(prompt);
        return response.toString();
      } catch (error) {
        return \`Error processing request: \${error}\`;
      }
    },
  },
});

if (import.meta.main) {
  app.run();
}
`);

In [ ]:
await writeFile("../backend/runtime/final_agent/identity_helper.ts", `/**
 * Helper for reading API keys back out of AgentCore Identity credential providers.
 *
 * Replaces \`identity_helper.py\`. The Gateway stores each target's API key in a credential provider
 * backed by Secrets Manager; the ExchangeRate spec wants the key as a request parameter, so the
 * agent has to read it back.
 */
import {
  BedrockAgentCoreControlClient,
  GetApiKeyCredentialProviderCommand,
  ListApiKeyCredentialProvidersCommand,
} from "@aws-sdk/client-bedrock-agentcore-control";
import { GetSecretValueCommand, SecretsManagerClient } from "@aws-sdk/client-secrets-manager";

export class IdentityHelper {
  readonly agentcoreClient: BedrockAgentCoreControlClient;
  readonly secretsClient: SecretsManagerClient;

  constructor(region = "us-east-1") {
    this.agentcoreClient = new BedrockAgentCoreControlClient({ region });
    this.secretsClient = new SecretsManagerClient({ region });
  }

  /** Find a credential provider by name prefix and return the API key it holds. */
  async getApiKeyByProviderName(providerNamePrefix: string): Promise<string | null> {
    try {
      // Step 1: find the provider
      const providers = await this.agentcoreClient.send(
        new ListApiKeyCredentialProvidersCommand({ maxResults: 100 }),
      );
      const provider = providers.credentialProviders?.find((p) =>
        p.name?.startsWith(providerNamePrefix)
      );
      if (!provider?.name) {
        console.log(\`❌ No credential provider found starting with '\${providerNamePrefix}'\`);
        return null;
      }

      // Step 2: read its details
      const details = await this.agentcoreClient.send(
        new GetApiKeyCredentialProviderCommand({ name: provider.name }),
      );

      // Step 3: extract secret ARN
      const secretArn = details.apiKeySecretArn?.secretArn;
      if (!secretArn) {
        console.log("❌ Credential provider has no secret ARN");
        return null;
      }
      console.log(\`   Secret ARN: \${secretArn}\`);

      // Step 4: read the secret
      const secret = await this.secretsClient.send(
        new GetSecretValueCommand({ SecretId: secretArn }),
      );
      const apiKey = parseSecretValue(secret.SecretString);
      if (!apiKey) {
        console.log("❌ Failed to parse API key from secret");
        return null;
      }
      return apiKey;
    } catch (error) {
      console.log(\`❌ Error retrieving API key: \${error}\`);
      return null;
    }
  }

  /** The ExchangeRate key, which its OpenAPI spec takes as a request parameter. */
  getExchangerateApiKey(): Promise<string | null> {
    return this.getApiKeyByProviderName("ExchangeRate-ApiKey");
  }
}

/**
 * Parse a secret value, accepting any of the field names AgentCore has used.
 * Exported for tests.
 */
export function parseSecretValue(secretString: string | undefined): string | null {
  if (!secretString) return null;
  try {
    const parsed = JSON.parse(secretString) as Record<string, unknown>;
    const key = parsed.api_key ?? parsed.apiKey ?? parsed.api_key_value ?? parsed.key;
    // If no standard field is found, hand back the whole JSON as the Python did
    return typeof key === "string" ? key : secretString;
  } catch {
    // Not JSON: the secret is the key itself
    return secretString;
  }
}
`);

## Step 3: Prepare Environment Variables for Runtime

In [ ]:
// Prepare environment variables from the loaded configurations.
// `Runtime.configure` takes them directly, so they are set as part of the deployment.

const MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0";

const runtimeEnvVars: Record<string, string> = {
  AWS_REGION: REGION,
  MODEL_ID,
};

// Add Gateway configuration
Object.assign(runtimeEnvVars, {
  GATEWAY_ID: gatewayInfo.gateway_id,
  GATEWAY_MCP_ENDPOINT: gatewayInfo.mcp_endpoint,
  GATEWAY_TOKEN_ENDPOINT: gatewayInfo.token_endpoint ?? cognitoConfig.client_info.token_endpoint,
  GATEWAY_OAUTH_CLIENT_ID: gatewayInfo.oauth_client_id,
  GATEWAY_OAUTH_CLIENT_SECRET: gatewayInfo.oauth_client_secret,
  GATEWAY_OAUTH_SCOPE: gatewayInfo.oauth_scope,
});
console.log("\u2705 Gateway environment variables prepared");

// Add Memory configuration
Object.assign(runtimeEnvVars, {
  MEMORY_ID: memoryInfo.memory_id,
  MEMORY_USER_ID: memoryInfo.user_id ?? "default-user",
  MEMORY_SESSION_ID: memoryInfo.session_id ?? "default-session",
});
console.log("\u2705 Memory environment variables prepared");

console.log(`\n\ud83d\udccb Total environment variables: ${Object.keys(runtimeEnvVars).length}`);
console.log("Environment variables ready for runtime deployment");

## Step 4: Create requirements file

On Deno the agent folder's own `deno.json` is the requirements file: it pins every dependency the
container installs, so it is what `requirementsFile` points at.

In [ ]:
await writeFile("../backend/runtime/final_agent/deno.json", `{
  "nodeModulesDir": "auto",
  "compilerOptions": {
    "strict": true
  },
  "imports": {
    "@strands-agents/sdk": "npm:@strands-agents/sdk@1.18.0",
    "bedrock-agentcore/": "npm:/bedrock-agentcore@0.4.4/",
    "zod": "npm:zod@4.6.5",
    "@opentelemetry/api": "npm:@opentelemetry/api@1.9.1",
    "@opentelemetry/api-logs": "npm:@opentelemetry/api-logs@0.219.0",
    "@opentelemetry/context-async-hooks": "npm:@opentelemetry/context-async-hooks@2.8.0",
    "@opentelemetry/core": "npm:@opentelemetry/core@2.8.0",
    "@opentelemetry/otlp-transformer": "npm:@opentelemetry/otlp-transformer@0.219.0",
    "@opentelemetry/resources": "npm:@opentelemetry/resources@2.8.0",
    "@opentelemetry/sdk-logs": "npm:@opentelemetry/sdk-logs@0.219.0",
    "@opentelemetry/sdk-trace-base": "npm:@opentelemetry/sdk-trace-base@2.8.0",
    "@smithy/protocol-http": "npm:@smithy/protocol-http@5.6.2",
    "@smithy/signature-v4": "npm:@smithy/signature-v4@5.7.3",
    "@aws-crypto/sha256-js": "npm:@aws-crypto/sha256-js@5.2.0",
    "@aws-sdk/credential-provider-node": "npm:@aws-sdk/credential-provider-node@3.972.83",
    "@aws-sdk/client-bedrock-agentcore": "npm:@aws-sdk/client-bedrock-agentcore@3.1136.0",
    "@aws-sdk/client-bedrock-agentcore-control": "npm:@aws-sdk/client-bedrock-agentcore-control@3.1136.0",
    "@aws-sdk/client-secrets-manager": "npm:@aws-sdk/client-secrets-manager@3.1136.0",
    "@std/path": "jsr:@std/path@1.1.6",
    "@std/dotenv": "jsr:@std/dotenv@0.225.8"
  }
}
`);

## Step 5: Configure Runtime Deployment

`Runtime.configure` takes `environmentVariables` directly, so the whole of the Python course's
"Step 7: Set Environment Variables Using AWS API" happens here — there is no second pass with
`update_agent_runtime` afterwards.

In [ ]:
import { Runtime } from "../toolkit/mod.ts";

const AGENT_NAME = "unified_travel_agent";
const SOURCE_DIR = "../backend/runtime/final_agent";

const agentcoreRuntime = new Runtime();

console.log("\ud83d\ude80 Configuring unified travel agent...");
console.log(`Environment variables to be set: ${Object.keys(runtimeEnvVars).length}`);

const configureResponse = await agentcoreRuntime.configure({
  entrypoint: "unified_travel_agent.ts",
  autoCreateExecutionRole: true,
  autoCreateEcr: true,
  requirementsFile: "deno.json",
  region: REGION,
  agentName: AGENT_NAME,
  sourceDir: SOURCE_DIR,
  environmentVariables: runtimeEnvVars,
});

console.log(`\u2705 Environment variables configured: ${Object.keys(runtimeEnvVars).length}`);
console.log("\u2705 Runtime configuration completed");
console.log(`Agent: ${configureResponse.agentName}`);
console.log(`Region: ${configureResponse.region}`);
console.log("Note: Memory ID is passed as an environment variable");

## Step 5: Deploy Unified Agent

In [ ]:
console.log("\ud83d\ude80 Deploying unified travel agent...");
console.log("This may take 5-10 minutes...");

const launchResult = await agentcoreRuntime.launch();

console.log("\u2705 Deployment completed!");
console.log(`Agent ARN: ${launchResult.agentArn}`);
console.log(`Agent ID: ${launchResult.agentId}`);
console.log(`ECR URI: ${launchResult.ecrUri}`);

// Save deployment info
const deploymentInfo = {
  agent_name: AGENT_NAME,
  agent_arn: launchResult.agentArn,
  agent_id: launchResult.agentId,
  ecr_uri: launchResult.ecrUri,
  region: REGION,
  integrated_resources: {
    gateway_id: gatewayInfo.gateway_id,
    memory_id: memoryInfo.memory_id,
    cognito_client_id: cognitoConfig.client_info.client_id,
  },
  capabilities: [
    "gateway_mcp_apis",
    "memory_preferences",
    "code_interpreter",
    "browser_tools",
    "unified_workflow",
  ],
};

// Saved next to the other notebooks' resource files, rather than into the agent source folder
await writeFile(
  "environments/final_deployment_info.json",
  `${JSON.stringify(deploymentInfo, null, 2)}\n`,
);
await state.set("final_deployment_info", deploymentInfo);

console.log("\ud83d\udcbe Deployment info saved to environments/final_deployment_info.json");

## Step 7: Set Environment Variables Using AWS API

Nothing to run here. The Python course needed a second pass with `update_agent_runtime` because its
starter toolkit could not take environment variables at configure time; ours can, so Step 5 already
sent `GATEWAY_*`, `MEMORY_*`, `MODEL_ID` and `AWS_REGION` with the runtime definition.

## Step 8: Verify Deployment Status

In [ ]:
console.log("\u23f3 Checking deployment status...");

let statusResponse = await agentcoreRuntime.status();
let status = (statusResponse.endpoint as { status?: string })?.status ?? "UNKNOWN";
const endStatus = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"];

while (!endStatus.includes(status)) {
  console.log(`Status: ${status}`);
  await new Promise((resolve) => setTimeout(resolve, 30_000));
  statusResponse = await agentcoreRuntime.status();
  status = (statusResponse.endpoint as { status?: string })?.status ?? "UNKNOWN";
}

console.log(`\n\ud83c\udf89 Final Status: ${status}`);
console.log(
  status === "READY"
    ? "\u2705 Unified travel agent is ready for testing!"
    : `\u274c Deployment failed with status: ${status}`,
);

### Update IAM Role to access AgentCore Identity and Secret Manager to retrieve ExchangeRate API Key

The Python course asks you to add these two statements to the execution role by hand. The course's
own `Runtime` helper already puts them in the role it creates (see `executionRolePolicy` in
`capstone_project/toolkit/runtime.ts`), so there is nothing to do — this is the policy it grants:

```JSON
{
			"Sid": "BedrockAgentCoreIdentityGetResourceApiKey",
			"Effect": "Allow",
			"Action": [
				"bedrock-agentcore:GetResourceApiKey",
				"bedrock-agentcore:ListApiKeyCredentialProviders",
				"bedrock-agentcore:GetApiKeyCredentialProvider"
			],
			"Resource": [
				"arn:aws:bedrock-agentcore:us-east-1:{YOU_AWS_ACCOUNT}:token-vault/default",
				"arn:aws:bedrock-agentcore:us-east-1:{YOU_AWS_ACCOUNT}:token-vault/default/apikeycredentialprovider/*",
				"arn:aws:bedrock-agentcore:us-east-1:{YOU_AWS_ACCOUNT}:workload-identity-directory/default",
				"arn:aws:bedrock-agentcore:us-east-1:{YOU_AWS_ACCOUNT}:workload-identity-directory/default/workload-identity/unified_travel_agent-*"
			]
		},
		{
			"Sid": "BedrockAgentCoreIdentityGetCredentialProviderClientSecret",
			"Effect": "Allow",
			"Action": [
				"secretsmanager:GetSecretValue"
			],
			"Resource": [
			    "arn:aws:secretsmanager:us-east-1:{YOU_AWS_ACCOUNT}:secret:bedrock-agentcore-identity!default/apikey/*",
				"arn:aws:secretsmanager:us-east-1:{YOU_AWS_ACCOUNT}:secret:bedrock-agentcore-identity!default/oauth2/*"
			]
		},
```

## Step 8: Test Unified Agent

In [ ]:
/** Invoke the unified travel agent and return its formatted response. */
async function invokeUnifiedAgent(runtime: Runtime, prompt: string): Promise<string> {
  try {
    console.log(`User: ${prompt.trim()}`);
    console.log("\nAgent Response:");
    console.log("-".repeat(60));

    // Invoke agent
    const invokeResponse = await runtime.invoke({ prompt });

    // The runtime answers with the agent's string, so no unescaping is needed
    const responseText = typeof invokeResponse === "string"
      ? invokeResponse
      : JSON.stringify(invokeResponse);

    console.log(responseText);
    console.log("-".repeat(60));

    return responseText;
  } catch (error) {
    const errorMsg = `\u274c Error invoking agent: ${error}`;
    console.log(errorMsg);
    return errorMsg;
  }
}

In [ ]:
await invokeUnifiedAgent(
  agentcoreRuntime,
  "What are flights available from JFK to FCO? The FlightSearch tool doenst need dates.",
);

In [ ]:
await invokeUnifiedAgent(agentcoreRuntime, "What is the conversion rate between \u20ac and $?");

In [ ]:
await invokeUnifiedAgent(agentcoreRuntime, "How is the Weather in Munich?");

In [ ]:
await invokeUnifiedAgent(
  agentcoreRuntime,
  "I want to travel to New York from Munich. " +
    "What are flight options, and how is currently the weather? " +
    "Also I want to exchange money in advance how much is 2000\u20ac in dollar currently? ",
);

## Step 9: Integration Summary

In [ ]:
// Load final deployment info
try {
  const finalInfo = JSON.parse(
    await Deno.readTextFile("environments/final_deployment_info.json"),
  ) as typeof deploymentInfo;

  console.log("\ud83c\udf89 UNIFIED TRAVEL AGENT DEPLOYMENT COMPLETE!");
  console.log("=".repeat(60));

  console.log("\n\ud83d\udccb Deployment Summary:");
  console.log(`  Agent Name: ${finalInfo.agent_name}`);
  console.log(`  Agent ID: ${finalInfo.agent_id}`);
  console.log(`  Region: ${finalInfo.region}`);

  console.log("\n\ud83d\udd17 Integrated Resources:");
  const integrated = finalInfo.integrated_resources;
  console.log(`  Gateway: ${integrated.gateway_id}`);
  console.log(`  Memory: ${integrated.memory_id}`);
  console.log(`  Cognito: ${integrated.cognito_client_id.slice(0, 10)}...`);

  console.log("\n\ud83d\udee0\ufe0f Unified Capabilities:");
  for (const capability of finalInfo.capabilities) {
    const title = capability.split("_").map((w) => w[0].toUpperCase() + w.slice(1)).join(" ");
    console.log(`  \u2705 ${title}`);
  }

  console.log("\n\ud83d\ude80 Complete Workflow Available:");
  console.log("  1. User Authentication (Cognito)");
  console.log("  2. Preference Retrieval (Memory)");
  console.log("  3. Flight/Hotel Search (Gateway MCP)");
  console.log("  4. Budget Analysis (Code Interpreter)");
  console.log("  5. Attraction Research (Browser Tools)");
  console.log("  6. Information Storage (Memory)");

  console.log("\n\u2705 All AgentCore components successfully integrated!");
  console.log("\u2705 No resource duplication - existing resources reused");
  console.log("\u2705 Production-ready unified travel agent deployed");
} catch (error) {
  console.log(`\u274c Error loading deployment info: ${error}`);
}

## Chapter 08 Complete!

### What We Accomplished

✅ **Unified Integration**: Combined all 7 AgentCore components into single agent
✅ **Resource Reuse**: Leveraged existing Gateway, Memory, and Cognito resources
✅ **No Duplication**: Avoided creating redundant AWS resources
✅ **Production Ready**: Deployed comprehensive travel agent to AgentCore Runtime
✅ **Environment Variables**: Configured with the runtime definition
✅ **End-to-End Workflow**: Complete travel planning from auth to storage

### Integrated Components

1. **Runtime**: Unified conversational agent deployment
2. **Gateway**: MCP integration for flights, hotels, weather, currency APIs
3. **Memory**: User preference storage and retrieval
4. **Identity**: Cognito authentication (reused from Chapter 03)
5. **Code Interpreter**: Budget calculations and data analysis
6. **Browser Tools**: Web research for attractions and reviews
7. **Observability**: Built-in monitoring and logging

### Complete Travel Planning Workflow

```
User Request → Authentication → Preference Retrieval → API Searches →
Budget Analysis → Attraction Research → Memory Storage → Final Itinerary
```